# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:
```text
1. [1, 2, 3] [0 1 1]
2. [4, 4 ,4]
3. [[0, 1, 2]
    [1, 2, 3]
    [2, 3, 4]]
    потому что броадкастинг дорисовывает обе матрицы в 3 на 3
    [0, 1, 2] становится [[0, 1, 2]
                          [0, 1, 2]
                          [0, 1, 2]]
     [[0] становится [[0, 0, 0]
     [1]              [1, 1, 1]
     [2]]             [2, 2, 2]]
4. 6
5. 1 + 4 + 9 + 16 = 30; 30/4=7.5
```

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Положите orders.csv в одну папку с ноутбуком
orders.csv на месте: False


In [3]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [4]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [5]:
# 1
print("Строк чистых", clean.sum())

september = clean & (month == '2026-09')
october = clean & (month == '2026-10')
print (amount[september].sum())
print (amount[october].sum())

difference = ( amount[october].sum() - amount[september].sum() ) / amount[september].sum()
print ( round( (difference * 100) , 2))

Строк чистых 4739
35243410.0
34419520.0
-2.34


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [6]:
# 2
cat1 = np.unique(cat)
matrix1 =np.zeros((5,2))

for i in range (len(cat1)):
    september = clean & (cat == cat1[i]) & (month == '2026-09')
    october = clean & (cat == cat1[i]) & (month == '2026-10')
    matrix1[i][0] = amount[september].sum()
    matrix1[i][1] = amount[october].sum()

sum_categories = matrix1.sum (axis = 1, keepdims = True)

print ("Больше всего выручки приносит -", cat1[sum_categories.argmax()])
difference = ( (matrix1[:, 1] - matrix1 [:, 0]) / matrix1 [:, 0] ) * 100

print ("Больше всего упала в выручке -", cat1[difference.argmin()] )

Больше всего выручки приносит - Электроника
Больше всего упала в выручке - Одежда


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [7]:
# 3
cat1 = np.unique(cat)

matrix1 =np.zeros((5,2))

for i in range (len(cat1)):
    september = clean & (cat == cat1[i]) & (month == '2026-09')
    october = clean & (cat == cat1[i]) & (month == '2026-10')
    matrix1[i][0] = amount[september].sum()
    matrix1[i][1] = amount[october].sum()


matrix2 = np.zeros((5,2))

for i in range (len(cat1)):
    september = clean & (cat == cat1[i]) & (month == '2026-09')
    october = clean & (cat == cat1[i]) & (month == '2026-10')
    matrix2[i][0] = len(amount[september])
    matrix2[i][1] = len(amount[october])


average = np.round(matrix1 / matrix2,2)
print(average)

print(f"Самый высокий средний чек в Октябре - {average[:, 1].max()}, у категории { cat1[average[:, 1].argmax()] } ")

[[10499.43 10768.47]
 [ 4020.    3815.75]
 [ 6187.09  6158.6 ]
 [13520.8  13045.6 ]
 [32772.75 33600.2 ]]
Самый высокий средний чек в Октябре - 33600.2, у категории Электроника 


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [8]:
# table
cities = np.unique(city)
months = np.array(["2026-09", "2026-10"])

# Таблица 6 × 2: строка = город, столбец = месяц
table = np.array([[amount[clean & (city == c) & (month == m)].sum() for m in months]
                  for c in cities])
for c, r in zip(cities, table):
    print(f"{c:<10} {r[0]:>12,.0f} {r[1]:>12,.0f}")
print(table.shape)

Актобе        2,576,960    2,540,420
Алматы       14,501,070   14,643,020
Астана        9,034,220    8,850,530
Атырау          923,990      969,860
Караганда     3,648,520    3,371,680
Шымкент       4,558,650    4,044,010
(6, 2)


In [9]:
# 4
sum1 = table.sum(axis = 0)
print ("Сумма по месяцам -", sum1)

print ("Доля каждого месяца")
percentage1 = table / sum1
print (np.round(percentage1, 2))
print (percentage1.sum(axis = 0))

index = list(cities).index("Шымкент")
difference = (percentage1[index, 1] - percentage1[index, 0]) * 100
print("Изменение доли Шымкента:", round(difference, 2),"%")

Сумма по месяцам - [35243410. 34419520.]
Доля каждого месяца
[[0.07 0.07]
 [0.41 0.43]
 [0.26 0.26]
 [0.03 0.03]
 [0.1  0.1 ]
 [0.13 0.12]]
[1. 1.]
Изменение доли Шымкента: -1.19 %


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [10]:
# 5
difference = table[:, 1] - table[:, 0]
total_difference = difference.sum()

index = list(cities).index("Шымкент")

contribution = (difference[index] / total_difference) * 100

print("Изменение выручки каждого города:", difference)
print("Общее изменение выручки:", total_difference)
print("Вклад Шымкента в падение:", round(contribution, 2), "%")

Изменение выручки каждого города: [ -36540.  141950. -183690.   45870. -276840. -514640.]
Общее изменение выручки: -823890.0
Вклад Шымкента в падение: 62.46 %


## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [11]:
# 6
share = table / table.sum (axis = 1, keepdims = True)

print("Доли сентября и октября для каждого города:")
print(np.round(share, 2))

Доли сентября и октября для каждого города:
[[0.5  0.5 ]
 [0.5  0.5 ]
 [0.51 0.49]
 [0.49 0.51]
 [0.52 0.48]
 [0.53 0.47]]


## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [ ]:
# 7

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [ ]:
# 8